In [238]:
from pathlib import Path

import polars as pl

ROOT = Path.cwd()
OUT_DIR = ROOT / "data" / "output"
if not OUT_DIR.exists():
    OUT_DIR = ROOT.parent / "data" / "output"

parquet_path = OUT_DIR / "forecast.parquet"
if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_path}")

raw_df = pl.read_parquet(parquet_path)

In [239]:
raw_df.head()

unique_id,ds,value,valuehat,y,yhat,train_start,train_end,test_start,test_end,forecast_start,forecast_end,sku_desc,store_name,seccion,period_type,driver_effect,driver_effect_value,yhat_seccion,valuehat_seccion,yhat_tienda,valuehat_tienda,modelo_seleccionado
str,date,f64,f64,f64,f64,date,date,date,date,date,date,str,str,str,str,f64,f64,f32,f32,f32,f32,str
"""1""",2024-05-26,2134653.8,2.0427e6,17188.0,16525.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null
"""1""",2024-05-27,1.9304e6,2.0117e6,16029.0,16911.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null
"""1""",2024-05-28,2.0906e6,2.2787e6,17250.0,18786.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null
"""1""",2024-05-29,1.8514e6,1.9099e6,16003.0,16741.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null
"""1""",2024-05-30,1.7116e6,1.8177e6,14036.0,15013.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null


In [240]:
df = raw_df.filter((pl.col("y") != 0)).with_columns(
    (pl.col("value") - pl.col("valuehat")).abs().alias("abs_err")
)["unique_id", "ds", "value", "valuehat", "abs_err", "period_type"]
print(df.head())

shape: (5, 6)
┌───────────┬────────────┬───────────┬──────────┬───────────┬─────────────┐
│ unique_id ┆ ds         ┆ value     ┆ valuehat ┆ abs_err   ┆ period_type │
│ ---       ┆ ---        ┆ ---       ┆ ---      ┆ ---       ┆ ---         │
│ str       ┆ date       ┆ f64       ┆ f64      ┆ f64       ┆ str         │
╞═══════════╪════════════╪═══════════╪══════════╪═══════════╪═════════════╡
│ 1         ┆ 2024-05-26 ┆ 2134653.8 ┆ 2.0427e6 ┆ 91914.67  ┆ in_sample   │
│ 1         ┆ 2024-05-27 ┆ 1.9304e6  ┆ 2.0117e6 ┆ 81269.57  ┆ in_sample   │
│ 1         ┆ 2024-05-28 ┆ 2.0906e6  ┆ 2.2787e6 ┆ 188084.34 ┆ in_sample   │
│ 1         ┆ 2024-05-29 ┆ 1.8514e6  ┆ 1.9099e6 ┆ 58553.66  ┆ in_sample   │
│ 1         ┆ 2024-05-30 ┆ 1.7116e6  ┆ 1.8177e6 ┆ 106086.21 ┆ in_sample   │
└───────────┴────────────┴───────────┴──────────┴───────────┴─────────────┘


In [241]:
sku_df = df.filter(
    (pl.col("unique_id") == "1||T:00001||S:46499")
    & (pl.col("period_type") == "in_sample")
)
print(sku_df.head())

shape: (5, 6)
┌─────────────────────┬────────────┬─────────┬──────────┬─────────┬─────────────┐
│ unique_id           ┆ ds         ┆ value   ┆ valuehat ┆ abs_err ┆ period_type │
│ ---                 ┆ ---        ┆ ---     ┆ ---      ┆ ---     ┆ ---         │
│ str                 ┆ date       ┆ f64     ┆ f64      ┆ f64     ┆ str         │
╞═════════════════════╪════════════╪═════════╪══════════╪═════════╪═════════════╡
│ 1||T:00001||S:46499 ┆ 2024-05-26 ┆ 8304.76 ┆ 8304.76  ┆ 0.0     ┆ in_sample   │
│ 1||T:00001||S:46499 ┆ 2024-05-27 ┆ 7575.47 ┆ 8972.06  ┆ 1396.59 ┆ in_sample   │
│ 1||T:00001||S:46499 ┆ 2024-05-28 ┆ 8146.44 ┆ 10044.87 ┆ 1898.43 ┆ in_sample   │
│ 1||T:00001||S:46499 ┆ 2024-05-29 ┆ 6897.0  ┆ 8221.93  ┆ 1324.93 ┆ in_sample   │
│ 1||T:00001||S:46499 ┆ 2024-05-30 ┆ 5935.89 ┆ 7202.67  ┆ 1266.78 ┆ in_sample   │
└─────────────────────┴────────────┴─────────┴──────────┴─────────┴─────────────┘


In [242]:
sku_df.count()

unique_id,ds,value,valuehat,abs_err,period_type
u32,u32,u32,u32,u32,u32
669,669,669,669,669,669


In [243]:
print(sku_df["value", "valuehat", "abs_err"].sum())
print(192969.01 / 632903.05)

shape: (1, 3)
┌──────────┬──────────┬───────────┐
│ value    ┆ valuehat ┆ abs_err   │
│ ---      ┆ ---      ┆ ---       │
│ f64      ┆ f64      ┆ f64       │
╞══════════╪══════════╪═══════════╡
│ 3.9079e6 ┆ 3.7061e6 ┆ 795821.43 │
└──────────┴──────────┴───────────┘
0.30489505462171496


In [244]:
wMAPE_sku = float(sku_df["abs_err"].sum()) / float(sku_df["value"].sum())
wMAPE_sku

0.20364267841325573

In [245]:
sto_df = df.filter(
    (pl.col("unique_id").str.contains("1||T:00001||S:", literal=True))
    & (pl.col("period_type") == "in_sample")
)
print(sto_df.head())

shape: (5, 6)
┌──────────────────────┬────────────┬───────┬──────────┬─────────┬─────────────┐
│ unique_id            ┆ ds         ┆ value ┆ valuehat ┆ abs_err ┆ period_type │
│ ---                  ┆ ---        ┆ ---   ┆ ---      ┆ ---     ┆ ---         │
│ str                  ┆ date       ┆ f64   ┆ f64      ┆ f64     ┆ str         │
╞══════════════════════╪════════════╪═══════╪══════════╪═════════╪═════════════╡
│ 1||T:00001||S:100175 ┆ 2025-02-06 ┆ 304.0 ┆ 0.69     ┆ 303.31  ┆ in_sample   │
│ 1||T:00001||S:100175 ┆ 2025-02-07 ┆ 456.0 ┆ 2.0      ┆ 454.0   ┆ in_sample   │
│ 1||T:00001||S:100175 ┆ 2025-02-08 ┆ 608.0 ┆ 4.22     ┆ 603.78  ┆ in_sample   │
│ 1||T:00001||S:100175 ┆ 2025-02-09 ┆ 304.0 ┆ 6.96     ┆ 297.04  ┆ in_sample   │
│ 1||T:00001||S:100175 ┆ 2025-02-11 ┆ 136.8 ┆ 9.73     ┆ 127.07  ┆ in_sample   │
└──────────────────────┴────────────┴───────┴──────────┴─────────┴─────────────┘


In [246]:
print(sto_df["value", "valuehat", "abs_err"].sum())

shape: (1, 3)
┌──────────┬──────────┬──────────┐
│ value    ┆ valuehat ┆ abs_err  │
│ ---      ┆ ---      ┆ ---      │
│ f64      ┆ f64      ┆ f64      │
╞══════════╪══════════╪══════════╡
│ 7.9480e8 ┆ 3.8231e8 ┆ 4.9837e8 │
└──────────┴──────────┴──────────┘


In [247]:
wMAPE_sto =  float(sto_df["abs_err"].sum()) / float(sto_df["value"].sum())
wMAPE_sto

0.6270426780066396

In [248]:
sec_df = df.filter(
    (pl.col("unique_id").str.count_matches(r"\|\|", literal=False) == 2)
    & (pl.col("period_type") == "in_sample")
)
print(sec_df.head())

shape: (5, 6)
┌──────────────────────┬────────────┬───────┬──────────┬─────────┬─────────────┐
│ unique_id            ┆ ds         ┆ value ┆ valuehat ┆ abs_err ┆ period_type │
│ ---                  ┆ ---        ┆ ---   ┆ ---      ┆ ---     ┆ ---         │
│ str                  ┆ date       ┆ f64   ┆ f64      ┆ f64     ┆ str         │
╞══════════════════════╪════════════╪═══════╪══════════╪═════════╪═════════════╡
│ 1||T:00211||S:100175 ┆ 2024-07-05 ┆ 157.0 ┆ 0.58     ┆ 156.42  ┆ in_sample   │
│ 1||T:00211||S:100175 ┆ 2024-11-01 ┆ 129.0 ┆ 0.52     ┆ 128.48  ┆ in_sample   │
│ 1||T:00211||S:100175 ┆ 2024-11-05 ┆ 129.0 ┆ 1.69     ┆ 127.31  ┆ in_sample   │
│ 1||T:00211||S:100175 ┆ 2025-01-14 ┆ 386.4 ┆ 1.07     ┆ 385.33  ┆ in_sample   │
│ 1||T:00211||S:100175 ┆ 2025-02-02 ┆ 138.0 ┆ 0.62     ┆ 137.38  ┆ in_sample   │
└──────────────────────┴────────────┴───────┴──────────┴─────────┴─────────────┘


In [249]:
wMAPE_sec = float(sec_df["abs_err"].sum()) / float(sec_df["value"].sum())
wMAPE_sec

0.6393224230704214